In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
readings = pd.read_csv("../data/parcel_readings.csv")
metadata = pd.read_csv("../data/parcel_metadata.csv")

In [5]:
print(readings.shape)
print(metadata.shape)

(3447, 6)
(28, 5)


In [6]:
readings.head()

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
0,PARCEL_018,16/05/2026,0.595,15.4,0.0,error
1,PARCEL_014,2026-01-27,0.457,27.6,0.0,OK
2,PARCEL_006,2026-05-17,0.497,25.8,0.0,OK
3,PARCEL_004,10/02/2026,0.810,25.0,0.0,OK
4,PARCEL_002,2026-01-17,0.269,33.2,0.0,OK


In [7]:
metadata.head()

,parcel_id,mill_id,crop_type,sowing_date,area_hectares
0,PARCEL_001,MILL_NORTH,sugarcane,2026-02-10,9.03
1,PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97
2,PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35
3,PARCEL_004,MILL_NORTH,sugarcane,2026-01-02,3.18
4,PARCEL_005,MILL_NORTH,soybean,2026-02-08,2.79


In [8]:
readings.info()

<class 'pandas.DataFrame'>
RangeIndex: 3447 entries, 0 to 3446
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      3447 non-null   str    
 1   date           3447 non-null   str    
 2   ndvi_value     3447 non-null   float64
 3   temperature_c  3447 non-null   float64
 4   rainfall_mm    3447 non-null   float64
 5   sensor_status  3310 non-null   str    
dtypes: float64(3), str(3)
memory usage: 161.7 KB


In [9]:
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      28 non-null     str    
 1   mill_id        28 non-null     str    
 2   crop_type      28 non-null     str    
 3   sowing_date    28 non-null     str    
 4   area_hectares  28 non-null     float64
dtypes: float64(1), str(4)
memory usage: 1.2 KB


In [10]:
readings.isnull().sum()

parcel_id          0
date               0
ndvi_value         0
temperature_c      0
rainfall_mm        0
sensor_status    137
dtype: int64

In [11]:
metadata.isnull().sum()

parcel_id        0
mill_id          0
crop_type        0
sowing_date      0
area_hectares    0
dtype: int64

In [12]:
readings.duplicated(
    subset=["parcel_id", "date"]
).sum()

np.int64(8)

In [13]:
invalid_ndvi = readings[
    ~readings["ndvi_value"].between(-1, 1)
]

invalid_ndvi.head()

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
21,PARCEL_017,2026-02-01,1.832,32.1,0.0,Error
72,PARCEL_001,2026-01-22,1.817,29.3,0.0,Error
113,PARCEL_013,2026-02-19,1.665,22.1,0.0,Error
128,PARCEL_024,2026-04-19,1.626,29.2,0.0,ERROR
151,PARCEL_015,10/05/2026,1.301,28.6,0.0,Error


In [14]:
len(invalid_ndvi)

104

In [15]:
readings["sensor_status"].value_counts(dropna=False)

sensor_status
OK       2890
NaN       137
Error      90
ERROR      80
error      76
ok         64
OK         57
 OK        53
Name: count, dtype: int64

In [16]:
pd.to_datetime(
    readings["date"],
    format="mixed",
    errors="coerce"
).isnull().sum()

np.int64(0)

In [17]:
metadata["parcel_id"].duplicated().sum()

np.int64(0)

In [18]:
metadata[metadata["area_hectares"] < 0]

,parcel_id,mill_id,crop_type,sowing_date,area_hectares


In [19]:

metadata["crop_type"].value_counts()

crop_type
sugarcane    20
soybean       6
wheat         2
Name: count, dtype: int64

In [20]:
merged = readings.merge(
    metadata,
    on="parcel_id",
    how="left"
)

merged.head()

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
0,PARCEL_018,16/05/2026,0.595,15.4,0.0,error,MILL_SOUTH,sugarcane,2026-01-11,5.82
1,PARCEL_014,2026-01-27,0.457,27.6,0.0,OK,MILL_NORTH,sugarcane,2026-01-05,9.39
2,PARCEL_006,2026-05-17,0.497,25.8,0.0,OK,MILL_WEST,sugarcane,2026-02-11,5.67
3,PARCEL_004,10/02/2026,0.810,25.0,0.0,OK,MILL_NORTH,sugarcane,2026-01-02,3.18
4,PARCEL_002,2026-01-17,0.269,33.2,0.0,OK,MILL_SOUTH,sugarcane,2026-01-16,8.97


In [21]:
merged["crop_type"].isnull().sum()

np.int64(40)

In [22]:
readings["ndvi_value"].describe()

count    3447.000000
mean        0.464235
std         0.349201
min        -1.976000
25%         0.278000
50%         0.502000
75%         0.653000
max         1.997000
Name: ndvi_value, dtype: float64